In [224]:
import pypsa
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import geopandas as gpd
from pypsa.plot import add_legend_lines, add_legend_patches, add_legend_semicircles
import yaml
from pathlib import Path
import pandas as pd
import yaml


**Set Up**

In [225]:
fn = 'resources/DE_test/networks/base_s_1__12h_2050.nc'


In [226]:
n= pypsa.Network(fn)

config = yaml.safe_load(Path("config/config.germany.yaml").read_text())


INFO:pypsa.network.io:New version 1.0.7 available! (Current: 0.35.2)
INFO:pypsa.network.io:Imported network 'Unnamed Network' has buses, carriers, generators, global_constraints, links, loads, storage_units, stores


In [227]:
p = Path(fn)  
try:
    if p.exists():
        p.unlink()
        print(f"Deleted {p}")
    else:
        print(f"File not found: {p}")
except Exception as e:
    print(f"Failed to delete {p}: {e}")

Deleted resources/DE_test/networks/base_s_1__12h_2050.nc


**Options**

In [228]:
ongrid=False
cluster_cost_reduction=1
cluster_size=1000   
renewables={"solar"}

In [229]:
nodes_with_clusters = n.buses.loc[
    n.buses.index.str[:2].isin(config['countries']) &
    (n.buses['carrier'] == 'AC')
].index.tolist()




In [230]:
nodes_with_clusters

['DE0 0']

**Buses and Generators of the Cluster Addition**

In [ ]:
def assign_cluster_generators_and_electricity_buses(n, config, cluster_size, cluster_cost_reduction, renewables):
    
    nodes_renewables_cf = {}                #dictionary of dataframes by node and renewable type, sorting the generators by average capacity factor (ascending order)
    clusters_generators={}                      #dictionary of dataframes by node and renewable type, containing the generators assigned to the cluster  

    for node in nodes_with_clusters:
        for renewable in renewables:

            nodes_renewables_cf[(node, renewable)] = pd.DataFrame(
                index=n.generators['p_nom_max'].loc[n.generators.index.astype(str).str.contains(rf"{node}.*{renewable}$")].index,
                columns=["p_max_pu","p_nom_max"]  
            )
            print(nodes_renewables_cf[(node, renewable)])

            clusters_generators[(node, renewable)] = pd.DataFrame()

            #we are considering the highest mean p_min_pu to determine the best generators per renewable available

            nodes_renewables_cf[(node, renewable)] ["p_max_pu"] = n.generators_t['p_max_pu'].loc[:, n.generators_t['p_max_pu'].columns.astype(str).str.contains(rf"{node}.*{renewable}$")].mean()
            nodes_renewables_cf[(node, renewable)] ["p_nom_max"] = n.generators['p_nom_max'].loc[n.generators.index.astype(str).str.contains(rf"{node}.*{renewable}$")]

            nodes_renewables_cf[(node, renewable)] = nodes_renewables_cf[(node, renewable)].sort_values("p_max_pu", ascending=False)

            print(nodes_renewables_cf[(node, renewable)])

            #print(nodes_renewables_cf[(country, renewable)])

            number_gen=0



            while  nodes_renewables_cf[(node, renewable)].iloc[0:number_gen+1]["p_nom_max"].sum() <= cluster_size:

                if number_gen >= len(nodes_renewables_cf[(node, renewable)]):
                    raise ValueError(f"Not enough {renewable} generators to reach cluster_size.")
                
                number_gen=number_gen+1

            #print(f"{renewable} generators in cluster: {number_gen+1}")

            clusters_generators[(node, renewable)]  = n.generators.loc[nodes_renewables_cf[(node, renewable)].index[0:number_gen+1]]
            remaining_capacity = nodes_renewables_cf[(node, renewable)].iloc[0:number_gen+1]["p_nom_max"].sum() - cluster_size
            #nodes_renewables_cf[(country, renewable)].iloc[number_gen]["p_nom_max"] = remaining_capacity maybe it is better to do this step later

            print(f"Remaining top {renewable} capacity outside the cluster: {remaining_capacity} MW")

            
            clusters_generators[(node, renewable)].loc[clusters_generators[(node, renewable)].index[number_gen], "p_nom_max"] = cluster_size - clusters_generators[(node, renewable)].loc[clusters_generators[(node, renewable)].index[0:number_gen],"p_nom_max"].sum()

            print(f"Capacity of the last {renewable} generator adjusted to fit cluster size: {clusters_generators[(node, renewable)].loc[clusters_generators[(node, renewable)].index[number_gen], 'p_nom_max']} MW")

            print(clusters_generators[(node, renewable)])

            for idx in clusters_generators[(node, renewable)].index:

                ### Electricity bus and generators ###

                if not n.buses.index.str.contains(rf"{clusters_generators[(node, renewable)].loc[idx].bus + " cluster"}$").any():
        
                    n.add(
                        "Bus",
                        name=clusters_generators[(node, renewable)].loc[idx].bus + " cluster",
                        v_nom=n.buses.at[clusters_generators[(node, renewable)].loc[idx].bus, "v_nom"],
                        x=n.buses.at[clusters_generators[(node, renewable)].loc[idx].bus, "x"],
                        y=n.buses.at[clusters_generators[(node, renewable)].loc[idx].bus, "y"],
                        unit=n.buses.at[clusters_generators[(node, renewable)].loc[idx].bus, "unit"],
                        location=n.buses.at[clusters_generators[(node, renewable)].loc[idx].bus, "location"],
                        country=n.buses.at[clusters_generators[(node, renewable)].loc[idx].bus, "country"],
                        carrier=n.buses.at[clusters_generators[(node, renewable)].loc[idx].bus, "carrier"],
                        control=n.buses.at[clusters_generators[(node, renewable)].loc[idx].bus, "control"],
                        substation_lv=n.buses.at[clusters_generators[(node, renewable)].loc[idx].bus, "substation_lv"],
                        substation_off=n.buses.at[clusters_generators[(node, renewable)].loc[idx].bus, "substation_off"],
                    )

                n.add(
                    "Generator",
                    name=clusters_generators[(node, renewable)].loc[idx].name + " cluster",
                    bus=clusters_generators[(node, renewable)].loc[idx].bus + " cluster",
                    carrier=clusters_generators[(node, renewable)].loc[idx].carrier,
                    p_nom_max=clusters_generators[(node, renewable)].loc[idx].p_nom_max,
                    p_max_pu=clusters_generators[(node, renewable)].loc[idx].p_max_pu,
                    marginal_cost=clusters_generators[(node, renewable)].loc[idx].marginal_cost*(1-cluster_cost_reduction),
                    capital_cost=clusters_generators[(node, renewable)].loc[idx].capital_cost*(1-cluster_cost_reduction),
                    efficiency=clusters_generators[(node, renewable)].loc[idx].efficiency,
                    p_nom_extendable=True,
                    p_nom_min=100,
                    overwrite=True,)


                
                
                #n.generators_t['p_max_pu'][clusters_generators[(node, renewable)].loc[idx].name + " cluster"] = n.generators_t['p_max_pu'][clusters_generators[(node, renewable)].loc[idx].name]


                ### H2 bus ##

                if not n.buses.index.str.contains(rf"{clusters_generators[(node, renewable)].loc[idx].bus + " H2 cluster"}$").any():

                    n.add(
                        "Bus",
                        name=clusters_generators[(node, renewable)].loc[idx].bus + " H2 cluster",
                        v_nom=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + " H2"}", "v_nom"],
                        x=n.buses.at[rf"{clusters_generators[(node,renewable)].loc[idx].bus + " H2"}", "x"],
                        y=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + " H2"}", "y"],
                        unit=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + " H2"}", "unit"],
                        location=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + " H2"}", "location"],
                        country=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + " H2"}", "country"],
                        carrier=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + " H2"}", "carrier"],
                        control=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + " H2"}", "control"],
                        substation_lv=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + " H2"}", "substation_lv"],
                        substation_off=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + " H2"}", "substation_off"],
                    )
                
                ### Batteries bus ###

                if not n.buses.index.str.contains(rf"{clusters_generators[(node, renewable)].loc[idx].bus + " battery cluster"}$").any():

                    n.add(
                        "Bus",
                        name=clusters_generators[(node, renewable)].loc[idx].bus + " battery cluster",
                        v_nom=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + " battery"}", "v_nom"],
                        x=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + " battery"}", "x"],
                        y=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + " battery"}", "y"],
                        unit=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + " battery"}", "unit"],
                        location=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + " battery"}", "location"],
                        country=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + " battery"}", "country"],
                        carrier=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + " battery"}", "carrier"],
                        control=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + " battery"}", "control"],
                        substation_lv=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + " battery"}", "substation_lv"],
                        substation_off=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + " battery"}", "substation_off"],
                    )



                if idx == nodes_renewables_cf[(node, renewable)].iloc[number_gen].name:
                    n.generators.loc[n.generators.index == idx, "p_nom_max"] = nodes_renewables_cf[(node, renewable)].iloc[0:number_gen+1]["p_nom_max"].sum() - cluster_size

                    print(f"Residual capacity of generator {clusters_generators[(node, renewable)].loc[idx].name} is {n.generators.loc[n.generators.index == idx, 'p_nom_max']} MW")
                
                else:


                    n.remove(
                            "Generator",
                            name=clusters_generators[(node, renewable)].loc[idx].name,
                    )



            

    return n

n= assign_cluster_generators_and_electricity_buses(n, config, cluster_size, cluster_cost_reduction, renewables)
            



            

            





        




              p_max_pu p_nom_max
Generator                       
DE0 0 0 solar      NaN       NaN
DE0 0 1 solar      NaN       NaN
DE0 0 2 solar      NaN       NaN
DE0 0 3 solar      NaN       NaN
DE0 0 4 solar      NaN       NaN
               p_max_pu      p_nom_max
Generator                             
DE0 0 4 solar  0.135553     269.893478
DE0 0 3 solar  0.124257    4633.581217
DE0 0 2 solar  0.117465  117200.434496
DE0 0 1 solar  0.110357  551477.060441
DE0 0 0 solar  0.105502  413112.136746
Remaining top solar capacity outside the cluster: 3903.4746951202105 MW
Capacity of the last solar generator adjusted to fit cluster size: 730.1065222128565 MW
                 bus control type       p_nom  p_nom_mod  p_nom_extendable  \
Generator                                                                    
DE0 0 4 solar  DE0 0      PQ         9.867500        0.0              True   
DE0 0 3 solar  DE0 0      PQ       156.204642        0.0              True   

                p_nom_m

In [232]:
n.generators_t['p_max_pu']

Generator,DE0 0 0 offwind-ac,DE0 0 0 offwind-dc,DE0 0 0 offwind-float,DE0 0 0 onwind,DE0 0 0 solar,DE0 0 0 solar rooftop,DE0 0 0 solar-hsat,DE0 0 1 onwind,DE0 0 1 solar,DE0 0 1 solar rooftop,...,DE0 0 3 solar,DE0 0 3 solar rooftop,DE0 0 3 solar-hsat,DE0 0 4 onwind,DE0 0 4 solar rooftop,DE0 0 4 solar-hsat,DE0 0 ror,DE0 0 rural solar thermal collector,DE0 0 urban central solar thermal collector,DE0 0 urban decentral solar thermal collector
snapshot,,,,,,,,,,,,,,,,,,,,,
2013-01-01 00:00:00,0.823409,0.640177,0.732674,0.089297,0.029343,0.029343,0.029065,0.566375,0.039824,0.039824,...,0.139634,0.139634,0.116876,0.781843,0.113277,0.124843,0.505006,0.003786,0.003786,0.003786
2013-01-01 12:00:00,0.669283,0.879109,0.688554,0.115275,0.011669,0.011669,0.010319,0.388776,0.016716,0.016716,...,0.084299,0.084299,0.064830,0.677654,0.066025,0.066151,0.541401,0.001199,0.001199,0.001199
2013-01-02 00:00:00,0.868284,0.884172,0.719742,0.061904,0.045240,0.045240,0.044073,0.304448,0.075627,0.075627,...,0.087375,0.087375,0.109048,0.874435,0.088623,0.096325,0.538470,0.010492,0.010492,0.010492
2013-01-02 12:00:00,0.875066,0.883960,0.767068,0.042344,0.020148,0.020148,0.017297,0.288079,0.034860,0.034860,...,0.050002,0.050002,0.039410,0.899125,0.047009,0.043344,0.526840,0.002425,0.002425,0.002425
2013-01-03 00:00:00,0.885489,0.885500,0.872462,0.125670,0.011598,0.011598,0.011822,0.582407,0.018740,0.018740,...,0.043038,0.043038,0.047075,0.998539,0.038478,0.048482,0.520168,0.000052,0.000052,0.000052
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2013-12-29 12:00:00,0.876791,0.880813,0.692393,0.064437,0.024074,0.024074,0.022726,0.246114,0.025976,0.025976,...,0.017794,0.017794,0.029633,0.928119,0.014305,0.014773,0.483989,0.001091,0.001091,0.001091
2013-12-30 00:00:00,0.870060,0.885500,0.794759,0.005990,0.102524,0.102524,0.099301,0.141410,0.113220,0.113220,...,0.159734,0.159734,0.153372,0.926800,0.131453,0.129511,0.480056,0.035658,0.035658,0.035658
2013-12-30 12:00:00,0.871186,0.884763,0.833278,0.014518,0.064506,0.064506,0.057724,0.210886,0.068913,0.068913,...,0.105433,0.105433,0.099853,0.983531,0.103312,0.105374,0.480705,0.016953,0.016953,0.016953


In [233]:
n.generators.loc[n.generators.index.str.contains('cluster')]

,bus,control,type,p_nom,p_nom_mod,p_nom_extendable,p_nom_min,p_nom_max,p_min_pu,p_max_pu,...,up_time_before,down_time_before,ramp_limit_up,ramp_limit_down,ramp_limit_start_up,ramp_limit_shut_down,weight,p_nom_opt,location,unit
Generator,,,,,,,,,,,,,,,,,,,,,
DE0 0 4 solar cluster,DE0 0 cluster,PQ,,0.0,0.0,True,800.0,269.893478,0.0,1.0,...,1,0,NaN,NaN,1.0,1.0,1.0,0.0,NaN,NaN
DE0 0 3 solar cluster,DE0 0 cluster,PQ,,0.0,0.0,True,800.0,730.106522,0.0,1.0,...,1,0,NaN,NaN,1.0,1.0,1.0,0.0,NaN,NaN


**Links of the Cluster Addition**

In [234]:
def add_cluster_links(n, nodes_with_clusters, cluster_cost_reduction, ongrid):

    for node in nodes_with_clusters:

        ### H2 Electrolysis ###

        link_name = f"{node} H2 Electrolysis"

        n.add(
            "Link",
            name=link_name + " cluster",
            bus0=n.links.at[link_name, "bus0"] + " cluster",
            bus1=n.links.at[link_name, "bus1"] + " cluster",
            p_nom_extendable=n.links.at[link_name, "p_nom_extendable"],
            carrier=n.links.at[link_name, "carrier"],
            efficiency=n.links.at[link_name, "efficiency"],
            capital_cost=n.links.at[link_name, "capital_cost"]*(1-cluster_cost_reduction),
            marginal_cost=n.links.at[link_name, "marginal_cost"]*(1-cluster_cost_reduction),
            lifetime=n.links.at[link_name, "lifetime"],
            reversed=False,
            overwrite=True,
        )

        ### Methanolization ###

        link_name = f"{node} methanolisation"

        n.add(
            "Link",
            name=link_name + " cluster",
            bus0=n.links.at[link_name, "bus0"] + " cluster",
            bus1=n.links.at[link_name, "bus1"],
            bus2=n.links.at[link_name, "bus2"] + " cluster",
            bus3=n.links.at[link_name, "bus3"],
            bus4=n.links.at[link_name, "bus4"],
            p_nom_extendable=n.links.at[link_name, "p_nom_extendable"],
            p_min_pu=n.links.at[link_name, "p_min_pu"],
            carrier=n.links.at[link_name, "carrier"],
            efficiency=n.links.at[link_name, "efficiency"],
            efficiency2=n.links.at[link_name, "efficiency2"],
            efficiency3=n.links.at[link_name, "efficiency3"],
            efficiency4=n.links.at[link_name, "efficiency4"],
            capital_cost=n.links.at[link_name, "capital_cost"]*(1-cluster_cost_reduction),
            marginal_cost=n.links.at[link_name, "marginal_cost"]*(1-cluster_cost_reduction),
            lifetime=n.links.at[link_name, "lifetime"],
            reversed=False,
            overwrite=True,
        )


    if ongrid==True :

        ### Electricity connection to grid ###

        link_name = f"{node} electricity cluster"
        
        n.add(
            "Link",
            name=link_name,
            bus0=f"{node} cluster",
            bus1=f"{node}",
            carrier=n.buses.at[f"{node}", "carrier"],  
            p_nom_extendable=True,
            efficiency=1.0,
            capital_cost=0.0,
            marginal_cost=0.0,
            reversed=False,
            overwrite=True,
        )

        link_name = f"{node} electricity cluster back"
        n.add(
            "Link",
            name=link_name,
            bus0=f"{node}",
            bus1=f"{node} cluster",
            carrier=n.buses.at[f"{node}", "carrier"],  
            p_nom_extendable=True,
            efficiency=1.0,
            capital_cost=0.0,
            marginal_cost=0.0,
            reversed=True,
            overwrite=True,
        )

    else:
        if f"{node} cluster electricity" in n.links.index:
            n.remove(
                "Link",
                name=f"{node} cluster electricity",
            )
        if f"{node} cluster electricity back" in n.links.index:
            n.remove(
                "Link",
                name=f"{node} cluster electricity back",
            )

    return n

n = add_cluster_links(n, nodes_with_clusters, cluster_cost_reduction, ongrid)



        


        

**Storages of the Cluster Addition**

In [235]:
def add_cluster_storages(n, nodes_with_clusters, cluster_cost_reduction):

    for node in nodes_with_clusters:

        link_name = f"{node} H2 Store"

    
        n.add("Store",
            name=link_name + " cluster",
            bus=n.stores.at[link_name, "bus"] + " cluster",
            carrier=n.stores.at[link_name, "carrier"],
            e_nom_extendable=True,
            capital_cost=n.stores.at[link_name, "capital_cost"]*(1-cluster_cost_reduction),
            marginal_cost=n.stores.at[link_name, "marginal_cost"]*(1-cluster_cost_reduction),
            e_initial_per_period=n.stores.at[link_name, "e_initial_per_period"],
            e_cyclic=n.stores.at[link_name, "e_cyclic"],
            e_cyclic_per_period=n.stores.at[link_name, "e_cyclic_per_period"],
            overwrite=True,
            )
        
        link_name = f"{node} battery"


        n.add(
                "Link",
                name=link_name + " charger cluster",
                bus0=f"{node} cluster",
                bus1=f"{node} battery cluster",
                carrier=n.buses.at[link_name, "carrier"],   
                p_nom_extendable=True,
                efficiency=1.0,
                capital_cost=0.0,
                marginal_cost=0.0,
                reversed=False,
                overwrite=True,
            )
        n.add(
                "Link",
                name=link_name + " discharger cluster",
                bus0=f"{node} battery cluster",
                bus1=f"{node} cluster",
                carrier=n.buses.at[link_name, "carrier"],
                p_nom_extendable=True,
                efficiency=1.0,
                capital_cost=0.0,
                marginal_cost=0.0,
                reversed=True,
                overwrite=True,
            )

        n.add("Store",
            name=link_name + " cluster" ,
            bus=f"{node} battery cluster",
            carrier=n.stores.at[link_name, "carrier"],
            e_nom_extendable=True,
            capital_cost=n.stores.at[link_name, "capital_cost"]*(1-cluster_cost_reduction),
            marginal_cost=n.stores.at[link_name, "marginal_cost"]*(1-cluster_cost_reduction),
            e_initial_per_period=n.stores.at[link_name, "e_initial_per_period"],
            e_cyclic=n.stores.at[link_name, "e_cyclic"],
            e_cyclic_per_period=n.stores.at[link_name, "e_cyclic_per_period"],
            overwrite=True,
            )
    return n

n = add_cluster_storages(n, nodes_with_clusters, cluster_cost_reduction)





In [236]:
n.links["reversed"] = n.links["reversed"].fillna(False).astype(bool)


**Printing to Check**

In [237]:
n.links.loc[n.links["bus1"]=='EU methanol']

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,efficiency4,bus3,efficiency3,bus2,efficiency2,tags,energy to power ratio,location,reversed,length_original
Link,,,,,,,,,,,,,,,,,,,,,
DE0 0 solid biomass biomass-to-methanol,DE0 0 solid biomass,EU methanol,,biomass-to-methanol,0.6500,True,0,20.0,0.0,0.0,...,1.000000,,1.000000,co2 atmosphere,-0.161400,NaN,NaN,,False,0.0
DE0 0 methanolisation,DE0 0 H2,EU methanol,,methanolisation,0.8787,True,0,20.0,0.0,0.0,...,0.021968,DE0 0 co2 stored,-0.217926,DE0 0,-0.238085,NaN,NaN,,False,0.0
DE0 0 methanolisation cluster,DE0 0 H2 cluster,EU methanol,,methanolisation,0.8787,True,0,20.0,0.0,0.0,...,0.021968,DE0 0 co2 stored,-0.217926,DE0 0 cluster,-0.238085,NaN,NaN,NaN,False,NaN


In [238]:
n.links.loc[n.links.index.str.contains("Electrolysis")]

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,efficiency4,bus3,efficiency3,bus2,efficiency2,tags,energy to power ratio,location,reversed,length_original
Link,,,,,,,,,,,,,,,,,,,,,
DE0 0 H2 Electrolysis,DE0 0,DE0 0 H2,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,1.0,,1.0,DE0 0 urban central heat,0.03515,NaN,NaN,,False,0.0
DE0 0 H2 Electrolysis cluster,DE0 0 cluster,DE0 0 H2 cluster,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,1.0,,1.0,,1.00000,NaN,NaN,NaN,False,NaN


In [239]:
n.links.loc[n.links["carrier"].str.contains('H2 Electrolysis')]

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,efficiency4,bus3,efficiency3,bus2,efficiency2,tags,energy to power ratio,location,reversed,length_original
Link,,,,,,,,,,,,,,,,,,,,,
DE0 0 H2 Electrolysis,DE0 0,DE0 0 H2,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,1.0,,1.0,DE0 0 urban central heat,0.03515,NaN,NaN,,False,0.0
DE0 0 H2 Electrolysis cluster,DE0 0 cluster,DE0 0 H2 cluster,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,1.0,,1.0,,1.00000,NaN,NaN,NaN,False,NaN


In [240]:
n.buses.loc[n.buses.index.str.contains("cluster")]

,v_nom,type,x,y,carrier,unit,location,v_mag_pu_set,v_mag_pu_min,v_mag_pu_max,control,generator,sub_network,country,substation_lv,substation_off
Bus,,,,,,,,,,,,,,,,
DE0 0 cluster,380.0,,9.212764,51.130936,AC,MWh_el,DE0 0,1.0,0.0,inf,Slack,,,DE,1.0,1.0
DE0 0 H2 cluster,1.0,,9.212764,51.130936,H2,MWh_LHV,DE0 0,1.0,0.0,inf,PQ,,,DE,NaN,NaN
DE0 0 battery cluster,1.0,,9.212764,51.130936,battery,MWh_el,DE0 0,1.0,0.0,inf,PQ,,,DE,NaN,NaN


In [241]:
n.stores.loc[n.stores.index.str.contains("cluster")]



,bus,type,carrier,e_nom,e_nom_mod,e_nom_extendable,e_nom_min,e_nom_max,e_min_pu,e_max_pu,...,marginal_cost,marginal_cost_quadratic,marginal_cost_storage,capital_cost,standing_loss,active,build_year,lifetime,e_nom_opt,location
Store,,,,,,,,,,,,,,,,,,,,,
DE0 0 H2 Store cluster,DE0 0 H2 cluster,,H2 Store,0.0,0.0,True,0.0,inf,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,True,0,inf,0.0,NaN
DE0 0 battery cluster,DE0 0 battery cluster,,battery,0.0,0.0,True,0.0,inf,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,True,0,inf,0.0,NaN


In [242]:
n.links.loc[n.links["carrier"]=='DC']

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,efficiency4,bus3,efficiency3,bus2,efficiency2,tags,energy to power ratio,location,reversed,length_original
Link,,,,,,,,,,,,,,,,,,,,,


In [243]:
n.stores

,bus,type,carrier,e_nom,e_nom_mod,e_nom_extendable,e_nom_min,e_nom_max,e_min_pu,e_max_pu,...,marginal_cost,marginal_cost_quadratic,marginal_cost_storage,capital_cost,standing_loss,active,build_year,lifetime,e_nom_opt,location
Store,,,,,,,,,,,,,,,,,,,,,
co2 atmosphere,co2 atmosphere,,co2,0.000000e+00,0.0,True,0.000000e+00,inf,-1.0,1.0,...,0.0,0.0,0.0,0.000000,0.000000,True,0,inf,0.0,
DE0 0 co2 stored,DE0 0 co2 stored,,co2 stored,0.000000e+00,0.0,True,0.000000e+00,inf,0.0,1.0,...,0.0,0.0,0.0,247.607546,0.000000,True,0,inf,0.0,
DE0 0 co2 sequestered,DE0 0 co2 sequestered,,co2 sequestered,0.000000e+00,0.0,True,0.000000e+00,7.914693e+07,0.0,1.0,...,-0.1,0.0,0.0,30.000000,0.000000,True,0,50.0,0.0,
DE0 0 gas Store,DE0 0 gas,,gas,0.000000e+00,0.0,True,2.433360e+08,inf,0.0,1.0,...,0.0,0.0,0.0,17.851178,0.000000,True,0,inf,0.0,
DE0 0 H2 Store,DE0 0 H2,,H2 Store,0.000000e+00,0.0,True,0.000000e+00,1.000000e+09,0.0,1.0,...,0.0,0.0,0.0,89.430064,0.000000,True,0,100.0,0.0,
DE0 0 battery,DE0 0 battery,,battery,0.000000e+00,0.0,True,0.000000e+00,inf,0.0,1.0,...,0.0,0.0,0.0,6427.168612,0.000000,True,0,30.0,0.0,
DE0 0 EV battery,DE0 0 EV battery,,EV battery,1.192899e+06,0.0,False,0.000000e+00,inf,0.0,1.0,...,0.0,0.0,0.0,0.000000,0.000000,True,0,inf,0.0,
DE0 0 urban central water tanks,DE0 0 urban central water tanks,,urban central water tanks,0.000000e+00,0.0,True,0.000000e+00,inf,0.0,1.0,...,0.0,0.0,0.0,258.096247,0.000077,True,0,40.0,0.0,
DE0 0 urban central water pits,DE0 0 urban central water pits,,urban central water pits,0.000000e+00,0.0,True,0.000000e+00,inf,0.0,1.0,...,0.0,0.0,0.0,81.683934,0.000078,True,0,30.0,0.0,


In [244]:
n.links.loc[n.links["bus0"]=='EU methanol']

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,efficiency4,bus3,efficiency3,bus2,efficiency2,tags,energy to power ratio,location,reversed,length_original
Link,,,,,,,,,,,,,,,,,,,,,
DE0 0 OCGT methanol,EU methanol,DE0 0,,OCGT methanol,0.43,True,0,25.0,0.0,0.0,...,1.0,,1.0,co2 atmosphere,0.24800,NaN,NaN,,False,0.0
EU industry methanol,EU methanol,EU industry methanol,,industry methanol,1.00,True,0,inf,0.0,0.0,...,1.0,,1.0,co2 atmosphere,0.24801,NaN,NaN,,False,0.0
EU shipping methanol,EU methanol,EU shipping methanol,,shipping methanol,1.00,True,0,inf,0.0,0.0,...,1.0,,1.0,co2 atmosphere,0.24801,NaN,NaN,,False,0.0


In [245]:
n.carriers

,co2_emissions,color,nice_name,max_growth,max_relative_growth
Carrier,,,,,
AC,0.0,#70af1d,AC,inf,0.0
offwind-dc,0.0,#74c6f2,Offshore Wind (DC),inf,0.0
offwind-ac,0.0,#6895dd,Offshore Wind (AC),inf,0.0
offwind-float,0.0,#b5e2fa,Offshore Wind (Floating),inf,0.0
onwind,0.0,#235ebc,Onshore Wind,inf,0.0
...,...,...,...,...,...
low-temperature heat for industry,0.0,#8f2727,low-temperature heat for industry,inf,0.0
industry electricity,0.0,#2d2a66,industry electricity,inf,0.0
H2 for industry,0.0,#f073da,H2 for industry,inf,0.0


In [246]:
n.global_constraints

,type,investment_period,carrier_attribute,sense,constant,mu
GlobalConstraint,,,,,,
lv_limit,transmission_volume_expansion_limit,NaN,"AC, DC",<=,0.000000e+00,0.0
biomass limit,operational_limit,NaN,solid biomass,<=,1.654047e+08,0.0
CO2Limit,co2_atmosphere,NaN,co2_emissions,<=,0.000000e+00,0.0


In [247]:
n.loads

,bus,carrier,type,p_set,q_set,sign,active
Load,,,,,,,
DE0 0,DE0 0 low voltage,electricity,,0.000000,0.0,-1.0,True
DE0 0 land transport EV,DE0 0 EV battery,land transport EV,,0.000000,0.0,-1.0,True
DE0 0 urban central heat,DE0 0 urban central heat,urban central heat,,0.000000,0.0,-1.0,True
DE0 0 solid biomass for industry,DE0 0 solid biomass for industry,solid biomass for industry,,15037.671233,0.0,-1.0,True
DE0 0 gas for industry,DE0 0 gas for industry,gas for industry,,5989.726027,0.0,-1.0,True
DE0 0 H2 for industry,DE0 0 H2,H2 for industry,,2128.995434,0.0,-1.0,True
EU industry methanol,EU industry methanol,industry methanol,,273.972603,0.0,-1.0,True
DE0 0 naphtha for industry,DE0 0 naphtha for industry,naphtha for industry,,8675.799087,0.0,-1.0,True
DE0 0 low-temperature heat for industry,DE0 0 urban central heat,low-temperature heat for industry,,1345.890411,0.0,-1.0,True


In [248]:
n.links

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,efficiency4,bus3,efficiency3,bus2,efficiency2,tags,energy to power ratio,location,reversed,length_original
Link,,,,,,,,,,,,,,,,,,,,,
DE0 0 co2 sequestered,DE0 0 co2 stored,DE0 0 co2 sequestered,,co2 sequestered,1.0000,True,0,inf,0.0,0.0,...,1.000000,,1.000000,,1.000000,NaN,NaN,,False,0.0
DE0 0 OCGT,DE0 0 gas,DE0 0,,OCGT,0.4300,True,0,25.0,0.0,0.0,...,1.000000,,1.000000,co2 atmosphere,0.198000,NaN,NaN,,False,0.0
DE0 0 CCGT,DE0 0 gas,DE0 0,,CCGT,0.6000,True,0,25.0,0.0,0.0,...,1.000000,,1.000000,co2 atmosphere,0.198000,NaN,NaN,,False,0.0
DE0 0 H2 Electrolysis,DE0 0,DE0 0 H2,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,1.000000,,1.000000,DE0 0 urban central heat,0.035150,NaN,NaN,,False,0.0
DE0 0 H2 Fuel Cell,DE0 0 H2,DE0 0,,H2 Fuel Cell,0.5000,True,0,10.0,0.0,0.0,...,1.000000,,1.000000,DE0 0 urban central heat,0.450000,NaN,NaN,,False,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
DE0 0 urban decentral water tanks discharger,DE0 0 urban decentral water tanks,DE0 0 urban decentral heat,,urban decentral water tanks discharger,1.0000,True,0,30.0,0.0,0.0,...,1.000000,,1.000000,,1.000000,NaN,NaN,,False,0.0
DE0 0 H2 Electrolysis cluster,DE0 0 cluster,DE0 0 H2 cluster,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,1.000000,,1.000000,,1.000000,NaN,NaN,NaN,False,NaN
DE0 0 methanolisation cluster,DE0 0 H2 cluster,EU methanol,,methanolisation,0.8787,True,0,20.0,0.0,0.0,...,0.021968,DE0 0 co2 stored,-0.217926,DE0 0 cluster,-0.238085,NaN,NaN,NaN,False,NaN


In [249]:
n.links.loc[n.links.index.str.contains("cluster"),n.links.columns.str.contains("bus")]

,bus0,bus1,bus4,bus3,bus2
Link,,,,,
DE0 0 H2 Electrolysis cluster,DE0 0 cluster,DE0 0 H2 cluster,,,
DE0 0 methanolisation cluster,DE0 0 H2 cluster,EU methanol,DE0 0 urban central heat,DE0 0 co2 stored,DE0 0 cluster
DE0 0 battery charger cluster,DE0 0 cluster,DE0 0 battery cluster,,,
DE0 0 battery discharger cluster,DE0 0 battery cluster,DE0 0 cluster,,,


In [250]:
n.generators.loc[n.generators.index.str.contains("solar")&
    ~n.generators.index.str.contains("solar-hsat") &~n.generators.index.str.contains("solar thermal") &~n.generators.index.str.contains("solar rooftop")]

,bus,control,type,p_nom,p_nom_mod,p_nom_extendable,p_nom_min,p_nom_max,p_min_pu,p_max_pu,...,up_time_before,down_time_before,ramp_limit_up,ramp_limit_down,ramp_limit_start_up,ramp_limit_shut_down,weight,p_nom_opt,location,unit
Generator,,,,,,,,,,,,,,,,,,,,,
DE0 0 0 solar,DE0 0,PQ,,16461.998596,0.0,True,16461.998596,413112.136746,0.0,1.0,...,1,0,NaN,NaN,1.0,1.0,1.0,0.0,,
DE0 0 1 solar,DE0 0,PQ,,31610.940093,0.0,True,31610.940093,551477.060441,0.0,1.0,...,1,0,NaN,NaN,1.0,1.0,1.0,0.0,,
DE0 0 2 solar,DE0 0,PQ,,5429.989169,0.0,True,5429.989169,117200.434496,0.0,1.0,...,1,0,NaN,NaN,1.0,1.0,1.0,0.0,,
DE0 0 3 solar,DE0 0,PQ,,156.204642,0.0,True,156.204642,3903.474695,0.0,1.0,...,1,0,NaN,NaN,1.0,1.0,1.0,0.0,,
DE0 0 4 solar cluster,DE0 0 cluster,PQ,,0.000000,0.0,True,800.000000,269.893478,0.0,1.0,...,1,0,NaN,NaN,1.0,1.0,1.0,0.0,NaN,NaN
DE0 0 3 solar cluster,DE0 0 cluster,PQ,,0.000000,0.0,True,800.000000,730.106522,0.0,1.0,...,1,0,NaN,NaN,1.0,1.0,1.0,0.0,NaN,NaN


In [251]:
n.generators.loc[n.generators.index.str.contains("cluster"), n.generators.columns.str.contains("bus")]

,bus
Generator,
DE0 0 4 solar cluster,DE0 0 cluster
DE0 0 3 solar cluster,DE0 0 cluster


In [252]:
n.generators_t['p_max_pu'].loc[:, n.generators_t['p_max_pu'].columns.str.contains("cluster")]

Generator
snapshot
2013-01-01 00:00:00
2013-01-01 12:00:00
2013-01-02 00:00:00
2013-01-02 12:00:00
2013-01-03 00:00:00
...
2013-12-29 12:00:00
2013-12-30 00:00:00
2013-12-30 12:00:00


**Exporting**

In [253]:
n.export_to_netcdf(fn)


INFO:pypsa.network.io:Exported network 'Unnamed Network'saved to 'resources/DE_test/networks/base_s_1__12h_2050.nc contains: loads, stores, carriers, links, global_constraints, generators, storage_units, buses


<xarray.Dataset> Size: 289kB
Dimensions:                               (snapshots: 730,
                                           investment_periods: 0, loads_i: 20,
                                           loads_t_p_set_i: 5, stores_i: 18,
                                           stores_t_e_min_pu_i: 1,
                                           stores_t_e_max_pu_i: 2,
                                           carriers_i: 113, links_i: 69,
                                           links_t_efficiency_i: 4,
                                           links_t_p_max_pu_i: 2,
                                           global_constraints_i: 3,
                                           generators_i: 36,
                                           generators_t_p_max_pu_i: 26,
                                           storage_units_i: 1, buses_i: 36)
Coordinates: (12/16)
  * snapshots                             (snapshots) int64 6kB 0 1 ... 728 729
  * investment_periods                    (investment_periods) object 0B 
  * loads_i                               (loads_i) object 160B 'DE0 0' ... '...
  * loads_t_p_set_i                       (loads_t_p_set_i) object 40B 'DE0 0...
  * stores_i                              (stores_i) object 144B 'co2 atmosph...
  * stores_t_e_min_pu_i                   (stores_t_e_min_pu_i) object 8B 'DE...
    ...                                    ...
  * links_t_p_max_pu_i                    (links_t_p_max_pu_i) object 16B 'DE...
  * global_constraints_i                  (global_constraints_i) object 24B '...
  * generators_i                          (generators_i) object 288B 'DE0 0 0...
  * generators_t_p_max_pu_i               (generators_t_p_max_pu_i) object 208B ...
  * storage_units_i                       (storage_units_i) object 8B 'DE0 0 ...
  * buses_i                               (buses_i) object 288B 'DE0 0' ... '...
Data variables: (12/92)
    snapshots_snapshot                    (snapshots) datetime64[ns] 6kB 2013...
    snapshots_objective                   (snapshots) float64 6kB 12.0 ... 12.0
    snapshots_stores                      (snapshots) float64 6kB 12.0 ... 12.0
    snapshots_generators                  (snapshots) float64 6kB 12.0 ... 12.0
    investment_periods_objective          (investment_periods) float64 0B 
    investment_periods_years              (investment_periods) float64 0B 
    ...                                    ...
    buses_unit                            (buses_i) object 288B 'MWh_el' ... ...
    buses_location                        (buses_i) object 288B 'DE0 0' ... '...
    buses_control                         (buses_i) object 288B 'Slack' ... 'PQ'
    buses_country                         (buses_i) object 288B 'DE' '' ... 'DE'
    buses_substation_lv                   (buses_i) float64 288B 1.0 nan ... nan
    buses_substation_off                  (buses_i) float64 288B 1.0 nan ... nan
Attributes:
    network__multi_invest:  0
    network_name:           Unnamed Network
    network_pypsa_version:  0.35.2
    network_srid:           4326
    crs:                    {"_crs": "GEOGCRS[\"WGS 84\",ENSEMBLE[\"World Geo...
    meta:                   {"version": "v2025.07.0", "tutorial": false, "log...